In [ ]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

data = gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt','w') as file:
  file.write(data)

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [ ]:
#DATA PREPROCESSING
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

with open('hamlet.txt','r') as file:
  text = file.read().lower()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
total_words

4818

In [ ]:
#creating index for words
tokenizer.word_index

In [ ]:
#creating input sequences
input_sequences = []
for line in text.split('\n'):
  token_list = tokenizer.texts_to_sequences([line])[0]

  for i in range(1, len(token_list)):
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

In [ ]:
input_sequences

In [ ]:
#pad ssequence
max_sequence_len = max([len(x) for x in input_sequences])
max_sequence_len

14

In [ ]:
input_sequences = np.array(pad_sequences(input_sequences, maxlen = max_sequence_len,padding='pre'))
input_sequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [ ]:
#train and test data
import tensorflow as tf
x,y = input_sequences[:,:-1], input_sequences[:,-1]


In [ ]:
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]], dtype=int32)

In [ ]:
y

array([ 687,    4,   45, ..., 1047,    4,  193], dtype=int32)

In [ ]:
y = tf.keras.utils.to_categorical(y,num_classes=total_words)

In [ ]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2)

In [ ]:
##training the LSTM RNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,Input

model = Sequential()

# model.add(Embedding(total_words,100,input_length=max_sequence_len))
model.add(Input(shape=(max_sequence_len-1,)))
model.add(Embedding(total_words, 100))
model.add(LSTM(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words,activation="softmax"))

##compile
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=['accuracy'])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 13, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    x_train,y_train,
    epochs=50,
    validation_data = (x_test,y_test),
    verbose=1
)

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 42s 58ms/step - accuracy: 0.0282 - loss: 7.1274 - val_accuracy: 0.0344 - val_loss: 6.7673
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 39s 60ms/step - accuracy: 0.0380 - loss: 6.4591 - val_accuracy: 0.0410 - val_loss: 6.8522
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 39s 60ms/step - accuracy: 0.0414 - loss: 6.3031 - val_accuracy: 0.0507 - val_loss: 6.9530
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 38s 58ms/step - accuracy: 0.0545 - loss: 6.1567 - val_accuracy: 0.0513 - val_loss: 6.9465
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 38s 58ms/step - accuracy: 0.0529 - loss: 6.0549 - val_accuracy: 0.0507 - val_loss: 7.0138
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 40s 62ms/step - accuracy: 0.0587 - loss: 5.9228 - val_accuracy: 0.0587 - val_loss: 6.9738
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 39s 59ms/step - accuracy: 0.0656 - loss: 5.7837 - val_accuracy: 0.0616 - val_loss: 7.0403
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 38s 59ms/step - accuracy: 0.0673 - loss: 5.6694 - 

In [22]:
history.history['accuracy']

[0.032499391585588455,
 0.03891183063387871,
 0.04619868844747543,
 0.05202817544341087,
 0.05484576150774956,
 0.05902355909347534,
 0.06563031673431396,
 0.06956521421670914,
 0.07563760131597519,
 0.08375030010938644,
 0.0908428430557251,
 0.09453485906124115,
 0.09997571259737015,
 0.10444498062133789,
 0.11066310107707977,
 0.115909643471241,
 0.12227349728345871,
 0.12980325520038605,
 0.1366529017686844,
 0.1491377204656601,
 0.16113674640655518,
 0.171144038438797,
 0.18664075434207916,
 0.20000000298023224,
 0.21161039173603058,
 0.22385232150554657,
 0.2371629774570465,
 0.24799610674381256,
 0.25955793261528015,
 0.26951664686203003,
 0.2819528877735138,
 0.29006558656692505,
 0.3051736652851105,
 0.3134806752204895,
 0.3200874328613281,
 0.3325236737728119,
 0.34112218022346497,
 0.34758320450782776,
 0.3581734299659729,
 0.36288559436798096,
 0.37726500630378723,
 0.3825601041316986,
 0.39091572165489197,
 0.3982025682926178,
 0.40723827481269836,
 0.4128248691558838,
 0.4

In [23]:
def predict_next_word(model, tokenizer, text, max_sequence_len):
  token_list = tokenizer.texts_to_sequences([text])[0]
  if len(token_list) >= max_sequence_len:
    token_list = token_list[-(max_sequence_len-1):]

  token_list = pad_sequences([token_list], maxlen=max_sequence_len-1,padding='pre')
  predicted = model.predict(token_list, verbose=0)
  predicted_word_index = np.argmax(predicted,axis=1)

  for word, index in tokenizer.word_index.items():
    if index == predicted_word_index:
      return word
  return None

In [24]:
input_text = "To be or not to be"
print(f"input text : {input_text}")
max_sequence_len = model.input_shape[1]+1
next_word = predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(next_word)

input text : To be or not to be
buried


In [25]:
model.save("next_word_lstm.h5")

In [26]:
import pickle
with open('tokenizer.pkl','wb') as handle:
  pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)